# UBBV | Rozwiązanie pokazowe | TensorFlow / Keras
Klasyfikacja doświadczenia i dominującego profilu kompetencyjnego. Dane syntetyczne, nie do decyzji kadrowych. Notebook odtwarza CSV, jeśli go nie ma. Wybór modelu na walidacji, test na końcu.

**Google Colab:** otwórz notebook i wybierz Uruchom wszystko. TensorFlow jest wymagany. Jeśli nie ma CSV, generator odtworzy dane. Wszystkie wyniki zostaną obliczone od nowa; nie są kopiowane z wersji PyTorch.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, classification_report
keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

## 1. Dane
Dwa ukryte czynniki generują obserwacje z szumem. Klasy są zbalansowane. Staż w latach, liczby projektów i szkoleń, umiejętności w skali 0–10. Celowo wprowadzamy 30 brakujących wartości.

In [ ]:
rng = np.random.default_rng(42)
N = 500
exp = rng.permutation(np.tile(np.arange(5), 100))
prof = rng.permutation(np.tile(np.arange(5), 100))
levels = ['Początkujący','Junior','Specjalista','Senior','Ekspert']
profiles = ['Techniczny','Analityczny','Organizacyjny','Menedżerski','Interdyscyplinarny']
staz = np.maximum(0, rng.normal(1+3*exp, 2.2)).round(1)
firm = np.minimum(staz, np.maximum(0, rng.normal(0.5+1.4*exp,1.5))).round(1)
means = np.array([[8,4,4,3],[4,8,4,3],[4,4,8,4],[4,4,6,8],[7,7,7,7]])
skills = np.clip(means[prof] + 0.35*(exp[:,None]-2) + rng.normal(0,1.5,(N,4)),0,10).round(1)
df = pd.DataFrame({'id': [f'UBBV-{i+1:04}' for i in range(N)],
 'dzial':rng.choice(['IT','Produkcja','Finanse','Logistyka','Administracja'],N),
 'staz_lata':staz,'staz_firma':firm,
 'projekty':rng.poisson(2+4*exp), 'szkolenia':rng.poisson(2+1.3*exp),
 'certyfikaty':rng.poisson(0.5+0.6*exp),
 'zlozonosc':np.clip(rng.normal(2+1.5*exp,1.8),0,10).round(1),
 'samodzielnosc':np.clip(rng.normal(2+1.6*exp,1.6),0,10).round(1),
 'techniczne':skills[:,0],'analityczne':skills[:,1], 'organizacyjne':skills[:,2], 'menedzerskie':skills[:,3],
 'zarzadzanie_lata':np.minimum(staz,np.maximum(0,rng.normal(exp*0.7+(prof==3)*2,1.5))).round(1),
 'doswiadczenie':np.array(levels)[exp], 'profil':np.array(profiles)[prof]})
for c in ['szkolenia','techniczne','samodzielnosc']:
    df.loc[rng.choice(N,10,replace=False),c] = np.nan
path = Path('UBBV_500_pracownikow.csv')
if path.exists():
    df = pd.read_csv(path)
else:
    df.to_csv(path,index=False)
assert len(df)==500 and df.id.is_unique
print(df.shape)
display(df.head(), df.isna().sum().to_frame('braki'), df[['doswiadczenie','profil']].apply(pd.Series.value_counts).fillna(0))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
df.groupby('doswiadczenie').staz_lata.mean().reindex(levels).plot.bar(ax=ax[0],title='Średni staż według klasy')
df.groupby('profil')[['techniczne','analityczne','organizacyjne','menedzerskie']].mean().reindex(profiles).plot.bar(ax=ax[1],title='Profile kompetencji')
plt.tight_layout(); plt.show()

## 2. Podział 350/75/75 i preprocessing
Usuwamy ID i etykiety z wejścia. Imputacja medianą, standaryzacja i one-hot dopasowane tylko na treningu. Stratyfikacja względem doświadczenia.

In [ ]:
X=df.drop(columns=['id','doswiadczenie','profil'])
y=np.column_stack([df.doswiadczenie.map(dict(zip(levels,range(5)))),df.profil.map(dict(zip(profiles,range(5))))]).astype('int64')
tr,rest=train_test_split(np.arange(len(df)),test_size=.30,random_state=42,stratify=y[:,0])
va,te=train_test_split(rest,test_size=.5,random_state=42,stratify=y[rest,0])
num=X.select_dtypes(include='number').columns.tolist()
prep=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),('cat',OneHotEncoder(handle_unknown='ignore'),['dzial'])])
def dense(a): return a.toarray() if hasattr(a,'toarray') else a
Xt=dense(prep.fit_transform(X.iloc[tr])).astype('float32')
Xv=dense(prep.transform(X.iloc[va])).astype('float32')
Xtest=dense(prep.transform(X.iloc[te])).astype('float32')
yt,yv,ys=[y[i] for i in [tr,va,te]]
print(Xt.shape,Xv.shape,Xtest.shape)

## 3. TensorFlow/Keras: sieć wielozadaniowa i trzy eksperymenty
Functional API: wspólne warstwy Dense/ReLU oraz dwie głowice Softmax po 5 klas. Adam, suma dwóch strat sparse categorical crossentropy. Wybór epoki według średniego macro F1 na walidacji.

In [ ]:
def build_model(widths, dropout):
    inputs=keras.Input(shape=(Xt.shape[1],),name='cechy')
    z=inputs
    for w in widths:
        z=keras.layers.Dense(w,activation='relu')(z)
        if dropout: z=keras.layers.Dropout(dropout)(z)
    a=keras.layers.Dense(5,activation='softmax',name='dosw')(z)
    b=keras.layers.Dense(5,activation='softmax',name='prof')(z)
    m=keras.Model(inputs,[a,b])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=.003),
              loss=['sparse_categorical_crossentropy']*2,
              metrics=['accuracy','accuracy'])
    return m

def probabilities(m,x):
    return [np.asarray(a) for a in m(x,training=False)]
def predict(m,x):
    return np.column_stack([a.argmax(1) for a in probabilities(m,x)])
def metrics(m,x,y):
    pred=predict(m,x)
    return {f'{name}_{metric}':fun(y[:,j],pred[:,j]) for j,name in enumerate(['dosw','prof']) for metric,fun in [('accuracy',accuracy_score),('F1',lambda a,b:f1_score(a,b,labels=range(5),average='macro',zero_division=0))]}

class BestF1(keras.callbacks.Callback):
    def __init__(self):
        super().__init__(); self.best=-1; self.rows=[]
    def on_epoch_end(self,epoch,logs=None):
        logs=logs or {}
        tm=metrics(self.model,Xt,yt); vm=metrics(self.model,Xv,yv)
        score=(vm['dosw_F1']+vm['prof_F1'])/2
        self.rows.append(dict(train_loss=logs['loss'],val_loss=logs['val_loss'],train_acc=(tm['dosw_accuracy']+tm['prof_accuracy'])/2,val_acc=(vm['dosw_accuracy']+vm['prof_accuracy'])/2))
        if score>self.best:
            self.best=score; self.weights=self.model.get_weights(); self.best_epoch=epoch+1

configs=[('A: 16',[16],0.,40),('B: 32-16',[32,16],.1,60),('C: 64-32-16',[64,32,16],.25,80)]
models={}; histories={}; results=[]
for name,widths,drop,epochs in configs:
    keras.utils.set_random_seed(42)
    m=build_model(widths,drop); cb=BestF1()
    m.fit(Xt,[yt[:,0],yt[:,1]],validation_data=(Xv,[yv[:,0],yv[:,1]]),epochs=epochs,batch_size=32,callbacks=[cb],verbose=0)
    m.set_weights(cb.weights); models[name]=m; histories[name]=pd.DataFrame(cb.rows)
    results.append({'model':name,'best_epoch':cb.best_epoch,**metrics(m,Xv,yv),'mean_F1':cb.best})
    print(name, 'najlepsze F1:',round(cb.best,3))
res=pd.DataFrame(results).sort_values('mean_F1',ascending=False)
display(res.round(3)); winner=res.iloc[0]['model']; model=models[winner]; print('Wybrany model:',winner)
model.summary()

In [ ]:
fig,axes=plt.subplots(3,2,figsize=(12,10))
for i,(name,h) in enumerate(histories.items()):
    h[['train_loss','val_loss']].plot(ax=axes[i,0],title=name+' – strata')
    h[['train_acc','val_acc']].plot(ax=axes[i,1],title=name+' – średnia accuracy')
plt.tight_layout(); plt.show()
res.set_index('model')[['dosw_F1','prof_F1']].plot.bar(title='Porównanie F1 na walidacji',ylim=(0,1)); plt.tight_layout(); plt.show()

## 4. Jednorazowy test zwycięskiego modelu
Macro F1 traktuje klasy równorzędnie. Baseline większościowy wyznaczamy na treningu.

In [ ]:
test_metrics=metrics(model,Xtest,ys); print(test_metrics)
for j in range(2):
    base=np.full(len(te),np.bincount(y[tr,j]).argmax()); print('Baseline',j,accuracy_score(y[te,j],base),f1_score(y[te,j],base,average='macro'))
pred=predict(model,Xtest)
fig,ax=plt.subplots(1,2,figsize=(13,5))
for j,names in enumerate([levels,profiles]):
    print(classification_report(y[te,j],pred[:,j],labels=range(5),target_names=names,zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y[te,j],pred[:,j],labels=range(5),display_labels=names,ax=ax[j],xticks_rotation=45,colorbar=False)
plt.tight_layout(); plt.show()

## 5. Pięć nowych osób
Ręcznie skonstruowane profile, bez etykiet wejściowych. Oczekiwania to hipotezy demonstracyjne, nie niezależna prawda referencyjna.

In [ ]:
new=pd.DataFrame([
['IT',0.8,.5,2,2,0,2,3,8,4,3,2,0],
['Finanse',4,2,7,4,1,4,5,4,9,4,3,0],
['Logistyka',7,4,13,5,2,5,6,4,4,9,4,1],
['Produkcja',11,6,17,7,3,8,8,4,5,7,9,5],
['Administracja',16,9,25,10,4,9,9,8,8,8,8,4]],columns=X.columns,index=['Ada','Bartek','Celina','Daniel','Ewa'])
display(new)
z=dense(prep.transform(new)).astype('float32')
pa,pb=probabilities(model,z)
output=pd.DataFrame({'doświadczenie':np.array(levels)[pa.argmax(1)],'P doświadczenie':pa.max(1),'profil':np.array(profiles)[pb.argmax(1)],'P profil':pb.max(1),'oczek. doświadczenie':levels,'oczek. profil':profiles},index=new.index)
display(output.round(3))
display(pd.DataFrame(pa,index=new.index,columns=levels).round(3),pd.DataFrame(pb,index=new.index,columns=profiles).round(3))
new.to_csv('UBBV_5_profili_testowych.csv',index_label='osoba')

## 6. Wnioski i ograniczenia

In [ ]:
print(f"Najlepszy na walidacji: {winner}; średnie macro F1: {res.iloc[0].mean_F1:.3f}")
print(f"Test: doświadczenie F1={test_metrics['dosw_F1']:.3f}; profil F1={test_metrics['prof_F1']:.3f}")
h=histories[winner]; k=int(res.iloc[0].best_epoch)-1
print(f"Różnica accuracy trening-walidacja w wybranej epoce: {h.iloc[k].train_acc-h.iloc[k].val_acc:.3f}")
base=res.iloc[0].mean_F1; effects=[]
for col in X.columns:
    altered=X.iloc[va].copy(); altered[col]=np.random.default_rng(7).permutation(altered[col].values)
    zz=dense(prep.transform(altered)).astype('float32')
    mm=metrics(model,zz,yv); effects.append((col,base-(mm['dosw_F1']+mm['prof_F1'])/2))
pd.DataFrame(effects,columns=['cecha','spadek_F1']).sort_values('spadek_F1',ascending=False).plot.bar(x='cecha',legend=False,title='Znaczenie cech: permutacja na walidacji'); plt.tight_layout(); plt.show()

**Interpretacja:** Większa sieć nie gwarantuje lepszej generalizacji. Duża dodatnia różnica trening–walidacja może sygnalizować przeuczenie. Permutacja pokazuje wrażliwość predykcji, nie przyczynowość. Softmax nie jest kalibrowaną pewnością. Model odtwarza zależności generatora, a nie rzeczywiste kompetencje ludzi. Realne badanie wymaga niezależnych etykiet, walidacji zewnętrznej i analizy sprawiedliwości. ChatGPT wspiera projekt i kod, człowiek odpowiada za definicje, założenia i interpretację. Eksperyment porównuje konfiguracje, nie izoluje wpływu jednego hiperparametru.

In [ ]:
# Zapis artefaktów do ponownego wykorzystania
import joblib, json
joblib.dump(prep,'UBBV_TF_preprocessing.joblib')
model.save('UBBV_model.keras')
Path('UBBV_TF_klasy.json').write_text(json.dumps({'dosw':levels,'prof':profiles},ensure_ascii=False),encoding='utf-8')
res.to_csv('UBBV_TF_wyniki_walidacji.csv',index=False)
output.to_csv('UBBV_TF_predykcje.csv',index_label='osoba')
# Kontrola zapisu i ponownego wczytania
restored=keras.models.load_model('UBBV_model.keras')
for a,b in zip(probabilities(model,z),probabilities(restored,z)):
    np.testing.assert_allclose(a,b,atol=1e-6)
print('Model zapisany i zweryfikowany po ponownym wczytaniu.')